## Conversational AI - Chatbots

In [1]:
# libraries
from dotenv import load_dotenv
import os
from openai import OpenAI
import gradio as gr

In [4]:
# set up environment
MODEL_GPT = 'gpt-4.1-mini'
MODEL_LLAMA = 'llama3.2'

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("API Keys not found")
elif not api_key.startswith('sk-proj-'):
    print("API key found but wrong format")
elif api_key.strip() != api_key:
    print("API key found but contain unnecessary space")
else:
    print("API Key found and in use")


openai = OpenAI()

OLLAMA_BASE_URL = 'http://localhost:11434/v1'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

API Key found and in use


In [5]:
system_message = "You are a helpful assistant."

In [6]:
# creating the callback function

def chat(message, history):
    return "Banana"


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [7]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say banana"


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    )

    response = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages)

    return response.choices[0].message.content


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [13]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    )

    stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)

    response = ''

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


In [14]:
# Using system message to add context and give an example answer - One-shot prompting

system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [15]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


In [16]:
def chat(message, history):
    history = [{'role': h["role"], 'content': h["content"]} for h in history]

    relevant_system_message = system_message

    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    messages = (
        [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]
    )

    stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)

    response = ''

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [17]:
gr.ChatInterface(fn=chat, type='messages').launch()

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.
